# OULAD — AN7: Outcome Associations

**Status:** package prepared; SQL not executed. Run cells individually. Do not use Run All. AN7.04 and AN7.05 are separate high-cost scans and must never run concurrently.

## 1. Purpose and boundaries

AN7 studies associations between recorded VLE activity and retrospective outcomes among registrations eligible at a defined landmark. It is not causal inference or predictive modeling. Primary features use post-registration exposure; course-relative measures remain separate for AN6 reconciliation. final_result is an outcome, never a predictor.

## 2. Frozen contract

- Grain: (code_module, code_presentation, id_student).
- L28 = days 0–27; L56 = 0–55; L84 = 0–83, inclusive.
- O1: Withdrawn vs other known final results among eligible registrations.
- O2: Fail vs Pass+Distinction among eligible non-withdrawn outcomes.
- O3: ordered Fail < Pass < Distinction among eligible non-withdrawn outcomes.
- The 93 Withdrawn/no-date cases have indeterminate landmark eligibility; the nine Fail/with-date cases remain visible for sensitivity.
- Active days are normalized by the number of post-registration days available before each landmark; raw counts and exposure duration remain available for review.
- Resource breadth is reported with exposure duration and requires registration-timing sensitivity because later registration shortens observation time.
- No activity truncation at date_unregistration; no source deduplication.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, spearmanr
from IPython.display import display, Markdown

_current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [_current, *_current.parents]
        if (candidate / "src" / "analytics" / "db.py").is_file()
        and (candidate / "sql" / "04_analytics").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from scripts.run_an7 import verify_read_only_preflight, run_an7_query, verify_cohort_gate

query_results = {}
KEY = ['code_module', 'code_presentation', 'id_student']
EXPECTED_REGISTRATIONS = 32_593
EXPECTED_DISTINCT_PEOPLE = 28_785
EXPECTED_ACTIVE_REGISTRATIONS = 28_500
EXPECTED_IN_COURSE_ROWS = 9_966_292

## 3. Read-only preflight

AN0.01 verifies database identity and both read-only settings before any AN7 SQL.

In [ ]:
preflight = verify_read_only_preflight()

## 4. AN7.01 — Outcome and registration reconciliation

Inspect all three result sets. Historical 93/9 conflict counts are references, not values to force.

In [ ]:
r = run_an7_query('AN7.01', max_rows=500, display_results=False)
query_results['AN7.01'] = r
assert r['result_set_count'] == 3 and not r['display_truncated']
print('AN7.01 completed:', r['elapsed_seconds'], 'seconds; result sets:', r['result_set_count'])

In [ ]:
display(query_results['AN7.01']['result_sets'][0]['data'])

### AN7.01 — Outcome and date-timing reconciliation

Compare final-result labels with registration and unregistration timing. Reconcile the historical 93 Withdrawn/no-date and nine non-Withdrawn/with-date cases, but do not force these values. The denominator is student-presentation rows, not distinct people.

**Review record:** capture the returned denominator, any deviation from historical evidence, the explanation supported by the output, and whether the cohort definition needs revision. No source value is changed.

In [ ]:
display(query_results['AN7.01']['result_sets'][1]['data'])

### AN7.01 — Presentation-level anomaly groups

Locate outcome/date conflicts by module-presentation and inspect their timing range. Keep final_result and date_unregistration separate. Do not relabel records or infer why a conflict occurred.

**Review record:** capture the returned denominator, any deviation from historical evidence, the explanation supported by the output, and whether the cohort definition needs revision. No source value is changed.

In [ ]:
display(query_results['AN7.01']['result_sets'][2]['data'])

### AN7.01 — Registration-date completeness

Reconcile 32,593 registration rows and 45 missing registration dates. Review negative, zero, and positive registration days by presentation. A negative day is course-relative and is not automatically an invalid registration.

**Review record:** capture the returned denominator, any deviation from historical evidence, the explanation supported by the output, and whether the cohort definition needs revision. No source value is changed.

## 5. AN7.02 — Assessment landmark context

Due-date/weight metadata does not prove score or feedback availability. Reuse AN4 saved outputs for submissions and scores.

In [ ]:
r = run_an7_query('AN7.02', max_rows=500, display_results=False)
query_results['AN7.02'] = r
assert r['result_set_count'] == 1 and not r['display_truncated']
print('AN7.02 completed:', r['elapsed_seconds'], 'seconds')

In [ ]:
display(query_results['AN7.02']['data'])

### AN7.02 — Assessment landmark context: interpretation

Read each row at module-presentation × assessment type × landmark grain. Assessments due by a landmark and their recorded weights describe stored metadata only. Check missing due dates, landmarks outside the course window, and variation in assessment structure. These fields do not establish when a score or feedback became visible to a student, and they do not authorize weighted-score or early-score features.

**Review record:** note assessment types due before each landmark, missing-date denominators, and presentations with no eligible landmark. Reuse saved AN4 submission/score evidence rather than rerunning AN4.

## 6. AN7.03 — Landmark cohort eligibility

This one-row-per-student-presentation result retains missing registration dates and undated Withdrawn cases as explicit statuses. The gate must pass before either heavy scan.

In [ ]:
r = run_an7_query('AN7.03', max_rows=33_000, display_results=False)
query_results['AN7.03'] = r
assert not r['display_truncated']
assert len(r['data']) == EXPECTED_REGISTRATIONS
assert not r['data'].duplicated(KEY).any()
for column in ('risk_status_l28', 'risk_status_l56', 'risk_status_l84'):
    print(f'\n{column}:')
    display(r['data'][column].value_counts(dropna=False).rename_axis('risk_status').to_frame('registrations'))
verify_cohort_gate(r)

### AN7.03 — Landmark eligibility: interpretation and approval gate

Each landmark has its own risk set. Review all status counts, not only AT_RISK. The primary cohort requires a matched registration and course window, a known registration date on/before the landmark, and no recorded unregistration on/before it. The 93 Withdrawn/no-date cases remain INDETERMINATE_WITHDRAWAL_TIMING where applicable; the 45 missing registration dates remain explicit.

**Acceptance checks:** 32,593 rows; unique three-column key; complete registration and course-window matches; 45 missing registration dates; and no silent status loss. Confirm that landmarks beyond a presentation's length are classified as LANDMARK_OUTSIDE_COURSE. The automated gate is necessary, not a substitute for analytical approval. Do not authorize either heavy scan until the status distributions and primary denominators have been reviewed.

## 7. AN7.04 — Daily activity extraction (high cost)

Run alone. Daily pre-aggregation reduces source rows to student-presentation × day before counting active days.

In [ ]:
r = run_an7_query('AN7.04', max_rows=33_000, display_results=False)
query_results['AN7.04'] = r
assert not r['display_truncated'], 'AN7.04 truncated; stop.'
assert len(r['data']) == EXPECTED_REGISTRATIONS
assert not r['data'].duplicated(KEY).any()
display(r['data'].head(10))
print('Rows:', len(r['data']), '| elapsed seconds:', r['elapsed_seconds'])
print('Course-active registrations:', int(r['data']['course_active_days_full'].gt(0).sum()))
print('In-course source activity rows:', int(r['data']['in_course_activity_rows_full'].sum()))

### AN7.04 — Daily activity: interpretation

This output separates course-relative active days from active days observed after personal registration. Python later divides post-registration active days by the available exposure days at each landmark. Review raw active-day counts and exposure denominators together.

**Mandatory reconciliation:** 32,593 unique student-presentations; 28,500 with in-course activity; 9,966,292 in-course source rows; and zero truncation. Check pre-course and in-course/pre-registration diagnostics separately. Negative course-relative days do not necessarily precede personal registration. Activity is not truncated at date_unregistration, and R0 source rows remain untouched.

**Performance:** record actual elapsed time and PostgreSQL diagnostics. If the query fails, stop and review the error; do not automatically retry or increase global memory settings.

## 8. Manual review gate — AN7.04

Inspect the complete AN7.04 frame and its retrieval metadata before allowing the second high-cost scan. This gate is intentionally manual; a successful SQL return alone is not analytical approval.

In [ ]:
AN7_04_REVIEWED = False
# After reviewing the complete AN7.04 output and its diagnostics, change this to True.
print('AN7.04 manual review approved:', AN7_04_REVIEWED)

**Manual approval checklist:** confirm the complete 32,593-row result was retrieved without truncation; the three-column key is unique; 28,500 activity-bearing registrations and 9,966,292 in-course rows reconcile to AN6; exposure-day values are plausible; and no unexpected date-boundary or retrieval anomalies remain. Change the flag to True only after this review.

## 8A. Refresh preflight and cohort gate before AN7.05

The second high-cost scan is a separate execution. Re-establish the read-only session and rerun AN7.01–AN7.03 so the cohort gate is fresh immediately before AN7.05.

In [ ]:
assert AN7_04_REVIEWED, 'AN7.04 must be reviewed before refreshing the gate.'
preflight = verify_read_only_preflight()
for query_id in ('AN7.01', 'AN7.02', 'AN7.03'):
    cap = 33_000 if query_id == 'AN7.03' else 500
    refreshed = run_an7_query(query_id, max_rows=cap, display_results=False)
    query_results[query_id] = refreshed
    assert not refreshed['display_truncated'], f'{query_id} truncated; stop.'
    print(query_id, '| result sets:', refreshed['result_set_count'], '| elapsed seconds:', refreshed['elapsed_seconds'])
    if query_id == 'AN7.03':
        verify_cohort_gate(refreshed)
print('Fresh AN7.01–AN7.03 cohort gate: PASS')

## 9. AN7.05 — Resource breadth extraction (high cost)

Run only after AN7.04 has been reviewed and the fresh AN0.01/AN7.01–AN7.03 gate has passed. This is a separate scan; never run it concurrently with AN7.04.

In [ ]:
r = run_an7_query('AN7.05', max_rows=33_000, display_results=False)
query_results['AN7.05'] = r
assert not r['display_truncated'], 'AN7.05 truncated; stop.'
assert len(r['data']) == EXPECTED_REGISTRATIONS
assert not r['data'].duplicated(KEY).any()
display(r['data'].head(10))
print('Rows:', len(r['data']), '| elapsed seconds:', r['elapsed_seconds'])
print('Course activity-bearing registrations with resource breadth:', int(r['data']['course_resource_breadth_full'].gt(0).sum()))
print('NULL resource-key groups:', int(r['data']['null_resource_key_groups'].sum()))

### AN7.05 — Resource breadth: interpretation

Resource breadth counts distinct id_site values first observed by each landmark. It is deliberately separate from active-day rates and is not divided by exposure days. Longer registration exposure creates more opportunity to encounter resources; interpret breadth alongside exposure duration and registration-timing sensitivity.

**Mandatory reconciliation:** 32,593 unique student-presentations; 28,500 with positive full-course resource breadth; zero NULL resource-key groups; no truncation; and non-decreasing cumulative breadth from L28 to L56 to L84 to full course. A positive breadth count means a recorded resource ID was observed, not that it was meaningfully used or learned from. R0 is preserved.

## 10. Merge and reconciliation

Merge only on the full three-column key with one-to-one validation. No association is calculated until the AN6 totals reconcile.

In [ ]:
cohort=query_results['AN7.03']['data'].copy()
daily=query_results['AN7.04']['data'].copy()
breadth=query_results['AN7.05']['data'].copy()
for name,frame in [('cohort',cohort),('daily',daily),('breadth',breadth)]:
    assert len(frame)==EXPECTED_REGISTRATIONS, f'{name}: unexpected row count'
    assert not frame.duplicated(KEY).any(), f'{name}: duplicate key'
analysis_df=cohort.merge(daily,on=KEY,how='left',validate='one_to_one')
analysis_df=analysis_df.merge(breadth,on=KEY,how='left',validate='one_to_one')
assert len(analysis_df)==EXPECTED_REGISTRATIONS
assert analysis_df.id_student.nunique()==EXPECTED_DISTINCT_PEOPLE
assert int(analysis_df.course_active_days_full.gt(0).sum())==EXPECTED_ACTIVE_REGISTRATIONS
assert int(analysis_df.course_resource_breadth_full.gt(0).sum())==EXPECTED_ACTIVE_REGISTRATIONS
assert int(analysis_df.in_course_activity_rows_full.sum())==EXPECTED_IN_COURSE_ROWS
assert int(analysis_df.null_resource_key_groups.sum())==0

# Normalize active days by the number of days each learner could actually be observed after registration.
for landmark, day in [("L28",27),("L56",55),("L84",83)]:
    exposure = analysis_df[f"registered_exposure_days_{landmark.lower()}"]
    active = analysis_df[f"registered_active_days_{landmark.lower()}"]
    analysis_df[f"registered_active_day_rate_{landmark.lower()}"] = (
        active / exposure.where(exposure.gt(0))
    )
analysis_df["registered_active_day_rate_full"] = (
    analysis_df["registered_active_days_full"]
    / analysis_df["registered_exposure_days_full"].where(
        analysis_df["registered_exposure_days_full"].gt(0)
    )
)

pairs=[('course_active_days_l28','course_active_days_l56'),('course_active_days_l56','course_active_days_l84'),('course_active_days_l84','course_active_days_full'),('registered_active_days_l28','registered_active_days_l56'),('registered_active_days_l56','registered_active_days_l84'),('registered_active_days_l84','registered_active_days_full'),('course_resource_breadth_l28','course_resource_breadth_l56'),('course_resource_breadth_l56','course_resource_breadth_l84'),('course_resource_breadth_l84','course_resource_breadth_full'),('registered_resource_breadth_l28','registered_resource_breadth_l56'),('registered_resource_breadth_l56','registered_resource_breadth_l84'),('registered_resource_breadth_l84','registered_resource_breadth_full')]
for left,right in pairs:
    assert (analysis_df[left]<=analysis_df[right]).all(), f'Non-monotone: {left}>{right}'
print('AN7 extraction reconciliation: PASS')
print('Registrations:',len(analysis_df),'Distinct people:',analysis_df.id_student.nunique())
print('In-course active:',int(analysis_df.course_active_days_full.gt(0).sum()))
print('In-course source rows:',int(analysis_df.in_course_activity_rows_full.sum()))

### Merge and reconciliation: interpretation

No association is interpretable until the one-to-one merges and all historical AN6 reconciliations pass. The full three-column key must remain unique in each input and after merging. Zero-inclusive registrations remain in the analysis frame. A missing activity row is not evidence of no learning. If any expected count differs, stop and investigate the join, date window, resource-key completeness, and retrieval before proceeding.

## 10A. Temporal leakage and feature-availability register

This register is a design control, not proof that every field is available in a live operational system. OULAD records day-level activity and retrospective outcomes; it does not provide event timestamps or score-publication timestamps. Features are restricted to recorded days through the inclusive landmark. A same-day operational decision requires an explicit end-of-day availability assumption.

| Field / measure | Analytical role | Availability basis | Leakage / interpretation risk | AN7 treatment |
|---|---|---|---|---|
| Post-registration active days and active-day rate | Primary activity feature | Recorded VLE days from GREATEST(0, date_registration) through L | Same-day event ordering is unknown | Include; retain raw days and exposure denominator |
| Post-registration resource breadth | Primary activity feature | Distinct resources first observed through L after personal registration | Exposure-opportunity bias | Include separately; show exposure and registration-timing sensitivity |
| Course-relative activity measures | Reconciliation diagnostics | Course-relative days 0 through L | Not personally registration-aligned | Keep separate from primary features |
| In-course source activity rows | Reconciliation diagnostic | All R0 rows in the course window | Repeated-row semantics unresolved | Never use as the primary feature |
| final_result | Outcome / target | Retrospective final course result | Direct target leakage if used as a predictor | O1/O2/O3 outcome only; never a feature |
| date_unregistration | Risk-set eligibility | Retrospective administrative date | Future information if treated as a feature | Eligibility/sensitivity only; never truncate activity |
| date_registration | Eligibility / exposure | Recorded personal registration day | Missingness and negative-day semantics | Define eligibility and exposure; not an unreviewed predictor |
| Assessment due dates and weights | Context metadata | Stored assessment metadata | Does not prove score/feedback availability | Describe only |
| student_assessment.score | Retrospective assessment result | Score visibility timestamp unavailable | High / unresolved | Exclude from early features |
| Banked date_submitted = -1 | Prior-attempt record | Sentinel, not a valid calendar day | Invalid timing interpretation | Exclude from timing features |
| Demographic / prior-study fields | Potential covariates | Profile fields; operational availability unverified | Confounding and availability need review | Do not include automatically |
| risk_status and id_student | Cohort selection / key | Derived status and identifier | Not predictive features | Cohort control and joins only |

In [ ]:
leakage_register = pd.DataFrame([
    {'field':'post-registration active days / active-day rate','role':'Primary activity feature','availability':'Recorded through inclusive landmark L after personal registration','risk':'Same-day event ordering is unavailable','treatment':'Include; retain raw days and exposure denominator'},
    {'field':'post-registration resource breadth','role':'Primary activity feature','availability':'Distinct resources first observed through L','risk':'Exposure-opportunity bias','treatment':'Include separately; registration-timing sensitivity'},
    {'field':'course-relative activity','role':'Reconciliation diagnostic','availability':'Course-relative days 0..L','risk':'Not personal-registration aligned','treatment':'Keep separate from primary features'},
    {'field':'in-course activity rows','role':'Reconciliation diagnostic','availability':'All R0 rows in course window','risk':'Repeated-row semantics unresolved','treatment':'Do not use as primary feature'},
    {'field':'final_result','role':'Outcome','availability':'Retrospective final result','risk':'Direct target leakage if used as feature','treatment':'Outcome only'},
    {'field':'date_unregistration','role':'Risk-set eligibility','availability':'Retrospective administrative date','risk':'Future information if treated as a feature','treatment':'Eligibility/sensitivity only; no activity truncation'},
    {'field':'date_registration','role':'Eligibility / exposure','availability':'Recorded registration day','risk':'Missingness and negative-day semantics','treatment':'Define eligibility and exposure'},
    {'field':'assessment due dates / weights','role':'Context metadata','availability':'Stored assessment metadata','risk':'Does not prove score availability','treatment':'Describe only'},
    {'field':'student_assessment.score','role':'Retrospective assessment result','availability':'Score visibility timestamp unavailable','risk':'High / unresolved','treatment':'Exclude from early features'},
    {'field':'banked date_submitted=-1','role':'Banked prior attempt','availability':'Sentinel, not a valid date','risk':'Invalid timing interpretation','treatment':'Exclude from timing features'},
    {'field':'demographic / prior-study fields','role':'Potential covariates','availability':'Operational availability unverified','risk':'Confounding / availability','treatment':'Do not include automatically'},
    {'field':'risk_status / id_student','role':'Selection state / key','availability':'Derived status and identifier','risk':'Not a model feature','treatment':'Cohort control and joins only'}
])
display(leakage_register)
assert all('final_result' not in cols for cols in [list(FEATURES.keys())])

## 11. Effect-size helpers

Binary outcomes use Cliff's delta. Confidence intervals resample student-presentations within each module-presentation; the grain contains one row per person within a presentation. Across-presentation results are not pooled as independent people.

In [ ]:
def cliffs_delta(x,y):
    x=pd.Series(x).dropna().to_numpy(); y=pd.Series(y).dropna().to_numpy()
    if not len(x) or not len(y): return np.nan
    u=mannwhitneyu(x,y,alternative='two-sided').statistic
    return float(2*u/(len(x)*len(y))-1)


def bootstrap_cliffs_ci(frame,feature,outcome_col,pos_label,neg_label,n_boot=500,seed=20261111):
    clean=frame[['id_student',feature,outcome_col]].dropna().copy()
    if clean.id_student.duplicated().any():
        raise ValueError('Bootstrap unit is not unique within this module-presentation.')
    positive=clean.loc[clean[outcome_col].eq(pos_label),feature].to_numpy()
    negative=clean.loc[clean[outcome_col].eq(neg_label),feature].to_numpy()
    n_positive,n_negative=len(positive),len(negative)
    if n_positive<10 or n_negative<10:
        return np.nan,np.nan,np.nan,n_positive,n_negative
    point=cliffs_delta(positive,negative)
    rng=np.random.default_rng(seed)
    boot=[]
    for _ in range(n_boot):
        x_sample=positive[rng.integers(0,n_positive,n_positive)]
        y_sample=negative[rng.integers(0,n_negative,n_negative)]
        boot.append(cliffs_delta(x_sample,y_sample))
    finite=np.asarray(boot,dtype=float)
    finite=finite[np.isfinite(finite)]
    if not len(finite):
        return point,np.nan,np.nan,n_positive,n_negative
    lo,hi=np.percentile(finite,[2.5,97.5])
    return point,float(lo),float(hi),n_positive,n_negative

LANDMARKS={'L28':'risk_status_l28','L56':'risk_status_l56','L84':'risk_status_l84'}
FEATURES={'active_day_rate':{'L28':'registered_active_day_rate_l28','L56':'registered_active_day_rate_l56','L84':'registered_active_day_rate_l84'},'resource_breadth':{'L28':'registered_resource_breadth_l28','L56':'registered_resource_breadth_l56','L84':'registered_resource_breadth_l84'}}
ORDINAL={'Fail':1,'Pass':2,'Distinction':3}

## 12. O1 and O2 — Binary associations by presentation and landmark

Positive Cliff's delta means the feature tends to be higher in the first-named outcome group. Confidence intervals are withheld when either outcome group has fewer than 10 observations; small cells remain visible in the descriptive output.

In [ ]:
binary_rows=[]
for landmark,risk_col in LANDMARKS.items():
    for (module,pres),group in analysis_df.groupby(['code_module','code_presentation']):
        eligible=group.loc[group[risk_col].eq('AT_RISK') & group.final_result.isin(['Withdrawn','Fail','Pass','Distinction'])].copy()
        for feature_name,feature_map in FEATURES.items():
            feature=feature_map[landmark]
            o1=eligible.copy(); o1['outcome']=np.where(o1.final_result.eq('Withdrawn'),'Withdrawn','Not_Withdrawn')
            delta,lo,hi,n_pos,n_neg=bootstrap_cliffs_ci(o1,feature,'outcome','Withdrawn','Not_Withdrawn')
            binary_rows.append({'outcome':'O1_WITHDRAWN','landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n_positive_feature_complete':n_pos,'n_negative_feature_complete':n_neg,'n_positive_outcome_total':int(o1.outcome.eq('Withdrawn').sum()),'n_negative_outcome_total':int(o1.outcome.eq('Not_Withdrawn').sum()),'effect_size':delta,'ci_low':lo,'ci_high':hi})
            o2=eligible.loc[eligible.final_result.isin(['Fail','Pass','Distinction'])].copy()
            o2['outcome']=np.where(o2.final_result.eq('Fail'),'Fail','Pass_or_Distinction')
            delta,lo,hi,n_pos,n_neg=bootstrap_cliffs_ci(o2,feature,'outcome','Fail','Pass_or_Distinction')
            binary_rows.append({'outcome':'O2_FAIL','landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n_positive_feature_complete':n_pos,'n_negative_feature_complete':n_neg,'n_positive_outcome_total':int(o2.outcome.eq('Fail').sum()),'n_negative_outcome_total':int(o2.outcome.eq('Pass_or_Distinction').sum()),'effect_size':delta,'ci_low':lo,'ci_high':hi})
binary_effects=pd.DataFrame(binary_rows)
display(binary_effects.head(20))

### O1/O2 effect estimates — interpretation

Each row is a module-presentation × landmark × feature comparison. Positive Cliff's delta means the feature tends to be higher in the first-named outcome group (Withdrawn for O1; Fail for O2). Review both feature-complete group sizes and total outcome-group sizes. Confidence intervals are withheld when either feature-complete group has fewer than 10 observations. The revised bootstrap resamples each outcome group separately, preserving its observed sample size and preventing empty-class replicates.

Do not interpret a positive/negative estimate as causal, and do not pool presentations as independent students. Compare effect direction and interval width across presentations.

## 13. Descriptive distributions for O1 and O2

Effect sizes are interpreted alongside group distributions. This table reports sample sizes, medians, and interquartile ranges by outcome, feature, landmark, and module-presentation. Exposure-day summaries must also be reviewed because registration timing changes the available observation period.

In [ ]:
distribution_rows=[]
for landmark,risk_col in LANDMARKS.items():
    for (module,pres),group in analysis_df.groupby(['code_module','code_presentation']):
        eligible=group.loc[group[risk_col].eq('AT_RISK') & group.final_result.isin(['Withdrawn','Fail','Pass','Distinction'])].copy()
        eligible['O1']=np.where(eligible.final_result.eq('Withdrawn'),'Withdrawn','Not_Withdrawn')
        o2=eligible.loc[eligible.final_result.isin(['Fail','Pass','Distinction'])].copy()
        o2['O2']=np.where(o2.final_result.eq('Fail'),'Fail','Pass_or_Distinction')
        for feature_name,feature_map in FEATURES.items():
            feature=feature_map[landmark]
            exposure_col=f'registered_exposure_days_{landmark.lower()}'
            for outcome_name,frame,outcome_col in [('O1',eligible,'O1'),('O2',o2,'O2')]:
                for group_name,part in frame.groupby(outcome_col):
                    values=part[feature].dropna()
                    distribution_rows.append({'outcome':outcome_name,'landmark':landmark,'code_module':module,
                        'code_presentation':pres,'feature':feature_name,'group':group_name,'n':len(values),
                        'median':values.median() if len(values) else np.nan,
                        'p25':values.quantile(.25) if len(values) else np.nan,
                        'p75':values.quantile(.75) if len(values) else np.nan,
                        'mean_exposure_days':part[exposure_col].mean()})
outcome_distributions=pd.DataFrame(distribution_rows)
display(outcome_distributions.head(30))

### O1/O2 distributions — interpretation

Use medians and interquartile ranges alongside effect sizes. The n column counts non-missing feature values in the outcome group; it is not necessarily the full outcome-group denominator. Compare mean exposure days between groups, especially for resource breadth and late registrants. Differences in exposure can explain part of a breadth contrast and must remain visible.

## 14. O3 — Ordinal associations

In [ ]:
ordinal_rows=[]
for landmark,risk_col in LANDMARKS.items():
    for (module,pres),group in analysis_df.groupby(['code_module','code_presentation']):
        eligible=group.loc[group[risk_col].eq('AT_RISK') & group.final_result.isin(ORDINAL)].copy()
        eligible['outcome_ordinal']=eligible.final_result.map(ORDINAL)
        for feature_name,feature_map in FEATURES.items():
            feature=feature_map[landmark]; subset=eligible[['id_student',feature,'outcome_ordinal']].dropna()
            if len(subset)<20 or subset[feature].nunique()<2 or subset.outcome_ordinal.nunique()<2:
                rho,lo,hi=np.nan,np.nan,np.nan
            else:
                rho=float(spearmanr(subset[feature],subset.outcome_ordinal).statistic)
                rng=np.random.default_rng(20261111); boot=[]
                for _ in range(300):
                    sample=subset.iloc[rng.integers(0,len(subset),len(subset))]
                    if sample[feature].nunique()>1 and sample.outcome_ordinal.nunique()>1:
                        boot.append(spearmanr(sample[feature],sample.outcome_ordinal).statistic)
                lo,hi=np.nanpercentile(boot,[2.5,97.5]) if boot else (np.nan,np.nan)
            ordinal_rows.append({'outcome':'O3_ORDINAL','landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n':len(subset),'spearman_rho':rho,'ci_low':lo,'ci_high':hi})
ordinal_effects=pd.DataFrame(ordinal_rows)
display(ordinal_effects.head(20))

### O3 ordinal association — interpretation

Spearman's rho evaluates monotonic ordering across Fail < Pass < Distinction; the numeric labels encode rank order only, not equal academic distances. Positive rho means higher feature values tend to accompany higher ordered outcomes. Review complete-record counts, ties and zero-heavy feature distributions, confidence-interval width, and presentation-specific direction. This remains a descriptive association.

## 15. Visual review of presentation-level associations

These boxplots summarize the distribution of presentation-specific effect estimates. Each presentation contributes one estimate; the figures are not pooled student-level effects and must not be read as a ranking of courses.

In [ ]:
for outcome_name, frame, value_col in [
    ('O1_WITHDRAWN', binary_effects, 'effect_size'),
    ('O2_FAIL', binary_effects, 'effect_size'),
    ('O3_ORDINAL', ordinal_effects, 'spearman_rho'),
]:
    for feature_name in FEATURES:
        subset=frame.loc[(frame.outcome.eq(outcome_name)) & frame.feature.eq(feature_name)].copy()
        if subset.empty:
            continue
        labels=[]; values=[]
        for landmark in LANDMARKS:
            series=subset.loc[subset.landmark.eq(landmark),value_col].dropna()
            if len(series):
                labels.append(landmark)
                values.append(series.to_numpy())
        if not values:
            continue
        fig,ax=plt.subplots(figsize=(8,4.5))
        ax.boxplot(values,labels=labels,showmeans=True)
        ax.axhline(0,linewidth=1)
        ax.set_title(f'{outcome_name}: presentation-level {feature_name} associations')
        ax.set_xlabel('Observation landmark')
        ax.set_ylabel(value_col)
        ax.grid(axis='y',alpha=.25)
        plt.tight_layout()
        plt.show()

## 16. Sensitivity analysis

Compare the primary O1 cohort with an alternative that treats undated Withdrawn cases as eligible. For O2, separately test exclusion of the nine Fail rows carrying an unregistration date. Source values are not relabelled.

In [ ]:
sensitivity=[]
for landmark,risk_col in LANDMARKS.items():
    for (module,pres),group in analysis_df.groupby(['code_module','code_presentation']):
        primary=group.loc[group[risk_col].eq('AT_RISK')].copy()
        alt=group.loc[group[risk_col].isin(['AT_RISK','INDETERMINATE_WITHDRAWAL_TIMING'])].copy()
        for feature_name,feature_map in FEATURES.items():
            feature=feature_map[landmark]
            for label,sample in [('O1_PRIMARY',primary),('O1_INCLUDE_UNDATED_WITHDRAWN',alt)]:
                sample=sample.loc[sample.final_result.isin(['Withdrawn','Fail','Pass','Distinction'])].copy()
                sample['outcome']=np.where(sample.final_result.eq('Withdrawn'),'Withdrawn','Not_Withdrawn')
                sensitivity.append({'sensitivity':label,'landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n':len(sample),'withdrawn_n':int(sample.outcome.eq('Withdrawn').sum()),'effect_size':cliffs_delta(sample.loc[sample.outcome.eq('Withdrawn'),feature],sample.loc[sample.outcome.eq('Not_Withdrawn'),feature])})
            o2=primary.loc[primary.final_result.isin(['Fail','Pass','Distinction'])].copy()
            o2['outcome']=np.where(o2.final_result.eq('Fail'),'Fail','Pass_or_Distinction')
            clean=o2.loc[~(o2.final_result.eq('Fail') & o2.date_unregistration.notna())]
            sensitivity.append({'sensitivity':'O2_EXCLUDE_FAIL_WITH_UNREGISTRATION_DATE','landmark':landmark,'code_module':module,'code_presentation':pres,'feature':feature_name,'n':len(clean),'fail_n':int(clean.outcome.eq('Fail').sum()),'effect_size':cliffs_delta(clean.loc[clean.outcome.eq('Fail'),feature],clean.loc[clean.outcome.eq('Pass_or_Distinction'),feature])})

            # Registration timing can shorten exposure and change resource opportunity.
            for timing_label, timing_mask in [
                ("REGISTERED_ON_OR_BEFORE_COURSE_START", primary.date_registration.le(0)),
                ("REGISTERED_AFTER_COURSE_START", primary.date_registration.gt(0)),
            ]:
                timing_sample=primary.loc[timing_mask].copy()
                timing_sample=timing_sample.loc[timing_sample.final_result.isin(["Withdrawn","Fail","Pass","Distinction"])]
                timing_sample["outcome"]=np.where(timing_sample.final_result.eq("Withdrawn"),"Withdrawn","Not_Withdrawn")
                sensitivity.append({"sensitivity":"O1_REGISTRATION_TIMING_STRATUM","registration_timing":timing_label,
                    "landmark":landmark,"code_module":module,"code_presentation":pres,"feature":feature_name,
                    "n":len(timing_sample),"withdrawn_n":int(timing_sample.outcome.eq("Withdrawn").sum()),
                    "effect_size":cliffs_delta(timing_sample.loc[timing_sample.outcome.eq("Withdrawn"),feature],
                                               timing_sample.loc[timing_sample.outcome.eq("Not_Withdrawn"),feature])})
sensitivity_effects=pd.DataFrame(sensitivity)
display(sensitivity_effects.head(20))

### Sensitivity results — interpretation

Compare the primary O1 risk set with the alternative including INDETERMINATE_WITHDRAWAL_TIMING, and compare O2 with/without the nine Fail/date-unregistration cases. Source labels are never changed. Review registration-timing strata for O1 activity rates and resource breadth; small strata may produce unstable estimates. If a sensitivity changes direction or materially changes magnitude, report that instability rather than selecting the preferred result.

## 17. Interpretation checkpoint

Review group sizes, missingness, effect direction/magnitude, bootstrap interval width, presentation heterogeneity, and 93/9 sensitivities. Do not rank courses or pool registrations as independent people. No result is causal or predictive.

In [ ]:
print('SQL queries executed:',sorted(query_results))
print('Analysis rows:',len(analysis_df))
print('Binary effect rows:',len(binary_effects))
print('Ordinal effect rows:',len(ordinal_effects))
print('Sensitivity rows:',len(sensitivity_effects))

## 18. AN7 closeout is not automatic

AN7 may be closed only after returned SQL outputs, reconciliation checks, effect-size tables, sensitivity analyses, and the leakage register have been reviewed. Until local execution outputs are returned and interpreted, this notebook contains no findings.

## 19. AN7 evidence report generation

Run this cell only after all required SQL outputs, reconciliation, association tables, and sensitivity analyses have been reviewed. It creates a local Markdown report under the Git-ignored outputs/an7 directory. The report summarizes quantitative evidence but leaves substantive interpretation and closure decisions for analyst review.

In [ ]:
required_queries = {'AN7.01', 'AN7.02', 'AN7.03', 'AN7.04', 'AN7.05'}
missing_queries = required_queries.difference(query_results)
if missing_queries:
    raise RuntimeError(f'Missing executed query outputs: {sorted(missing_queries)}')
if not all(name in globals() for name in ('analysis_df','binary_effects','outcome_distributions','ordinal_effects','sensitivity_effects','leakage_register')):
    raise RuntimeError('Complete merge, statistical analyses, sensitivity analysis, and leakage register first.')

# Summarize presentation-level estimates without pooling student-presentations.
summary_rows=[]
for outcome_name, frame, value_col in [
    ('O1_WITHDRAWN', binary_effects, 'effect_size'),
    ('O2_FAIL', binary_effects, 'effect_size'),
    ('O3_ORDINAL', ordinal_effects, 'spearman_rho'),
]:
    for (landmark, feature), group in frame.loc[frame.outcome.eq(outcome_name)].groupby(['landmark','feature']):
        valid=group[value_col].dropna()
        ci_valid=group.dropna(subset=['ci_low','ci_high'])
        summary_rows.append({
            'outcome':outcome_name,'landmark':landmark,'feature':feature,
            'presentations_with_estimate':int(len(valid)),
            'positive_estimates':int((valid>0).sum()),
            'negative_estimates':int((valid<0).sum()),
            'median_presentation_effect':float(valid.median()) if len(valid) else np.nan,
            'CIs_wholly_above_zero':int((ci_valid.ci_low>0).sum()),
            'CIs_wholly_below_zero':int((ci_valid.ci_high<0).sum()),
        })
effect_summary=pd.DataFrame(summary_rows)

reconciliation_rows=[
    ('Registered student-presentations',len(analysis_df),EXPECTED_REGISTRATIONS),
    ('Distinct people across presentations',int(analysis_df.id_student.nunique()),EXPECTED_DISTINCT_PEOPLE),
    ('In-course activity-bearing registrations',int(analysis_df.course_active_days_full.gt(0).sum()),EXPECTED_ACTIVE_REGISTRATIONS),
    ('Full-course resource-bearing registrations',int(analysis_df.course_resource_breadth_full.gt(0).sum()),EXPECTED_ACTIVE_REGISTRATIONS),
    ('In-course source activity rows',int(analysis_df.in_course_activity_rows_full.sum()),EXPECTED_IN_COURSE_ROWS),
    ('NULL resource-key groups',int(analysis_df.null_resource_key_groups.sum()),0),
]
reconciliation_table=pd.DataFrame(reconciliation_rows,columns=['check','observed','historical_reference'])

query_rows=[]
for query_id in sorted(query_results):
    result=query_results[query_id]
    for item in result['result_sets']:
        query_rows.append({'query_id':query_id,'result_set':item['statement_number'],
            'rows_returned':item['rows_displayed'],'truncated':item['display_truncated'],
            'batch_seconds':result['elapsed_seconds']})
query_table=pd.DataFrame(query_rows)

research_status=pd.DataFrame([
    {'research_question':'O1 — Withdrawn versus other known outcomes','status':'Quantitatively evaluated; interpretation review required' if (binary_effects.outcome=='O1_WITHDRAWN').any() else 'Not evaluated'},
    {'research_question':'O2 — Fail versus Pass/Distinction','status':'Quantitatively evaluated; interpretation review required' if (binary_effects.outcome=='O2_FAIL').any() else 'Not evaluated'},
    {'research_question':'O3 — Fail < Pass < Distinction','status':'Quantitatively evaluated; interpretation review required' if (ordinal_effects.outcome=='O3_ORDINAL').any() else 'Not evaluated'},
    {'research_question':'Presentation and landmark heterogeneity','status':'Described by presentation-specific estimates' if len(effect_summary) else 'Not evaluated'},
    {'research_question':'93/9 outcome-date sensitivity','status':'Sensitivity rows generated; interpretation review required' if len(sensitivity_effects) else 'Not evaluated'},
    {'research_question':'Assessment due-date/weight context','status':'Metadata returned; no score-availability inference' if 'AN7.02' in query_results else 'Not evaluated'},
])

AN7_DECISION_NOTES = {
    'O1 interpretation and decision':'Pending analyst review of effect sizes, intervals, denominators, and presentation heterogeneity.',
    'O2 interpretation and decision':'Pending analyst review of effect sizes, intervals, denominators, and the nine-case sensitivity.',
    'O3 interpretation and decision':'Pending analyst review of rank correlations, intervals, ties, and presentation heterogeneity.',
    'Limitations and unresolved questions':'Complete after reviewing missingness, exposure differences, same-day timing, confounding, and source semantics.',
    'AN8 recommendation':'Carry forward only candidates with defensible cutoffs, stable presentation-level evidence, useful uncertainty, and documented availability. AN8 remains a readiness gate; no model is authorized here.'
}

def _md_table(frame):
    if frame.empty:
        return '_No rows available._'
    headers=[str(x) for x in frame.columns]
    rows=[]
    for row in frame.itertuples(index=False,name=None):
        rows.append([str(x) if pd.notna(x) else '—' for x in row])
    lines=['| '+' | '.join(headers)+' |','| '+' | '.join(['---']*len(headers))+' |']
    lines.extend('| '+' | '.join(v.replace('|','\\|') for v in row)+' |' for row in rows)
    return '\n'.join(lines)

report_lines=[
    '# OULAD — AN7 Outcome Associations: Evidence Report',
    '',
    f'Generated (UTC): {datetime.now(timezone.utc).isoformat()}',
    '',
    '**Evidence status:** locally executed outputs retained in this notebook; interpretation and closure require analyst approval.',
    '',
    '## 1. Execution record',
    _md_table(query_table),
    '',
    '## 2. Reconciliation against AN6',
    _md_table(reconciliation_table),
    '',
    '## 3. Research questions and analytical status',
    _md_table(research_status),
    '',
    '## 4. Presentation-level effect summary',
    'These summaries count presentation-specific directions; they are not pooled student-level effects or presentation rankings.',
    _md_table(effect_summary),
    '',
    '## 5. Sensitivity coverage',
    f'- Sensitivity rows generated: {len(sensitivity_effects)}.',
    f'- O1/O2 feature-distribution rows: {len(outcome_distributions)}.',
    '- Review the 93 undated Withdrawn cases and nine Fail/with-unregistration-date cases separately; do not relabel source outcomes.',
    '',
    '## 6. Temporal leakage controls',
    _md_table(leakage_register),
    '',
    '## 7. Findings and decisions requiring analyst sign-off',
]
for label,note in AN7_DECISION_NOTES.items():
    report_lines.extend([f'### {label}',note,''])
report_lines.extend([
    '## 8. Limitations',
    '- Associations are observational and do not establish causality or predictive performance.',
    '- Repeated student identities across presentations require presentation-aware interpretation; no naïve pooled independence assumption is made.',
    '- Resource breadth is exposure-sensitive and is not normalized by available days.',
    '- VLE dates are day-level; within-day event availability is not observed.',
    '- Score/feedback publication times are unavailable; assessment scores are excluded from early features.',
    '- R0 preserves repeated source rows; row and click totals remain sensitive to unresolved source repetition.',
    '- The 93 undated Withdrawn and nine Fail/date-unregistration cases remain source anomalies for sensitivity review.',
    '',
    '## 9. AN8 handover recommendations',
    '- Treat AN8 as a predictive-readiness gate, not automatic model authorization.',
    '- Define a specific operational decision cutoff and verify every candidate feature was available by that cutoff.',
    '- Carry forward only features with defensible denominators, acceptable missingness, stable presentation-level evidence, and interpretable uncertainty.',
    '- Specify temporal validation across presentations before considering any model; do not use a random split by student-presentation.',
    '- Reassess confounding, repeated-student clustering, class imbalance, calibration, and subgroup limitations before predictive implementation.',
    '',
    '## 10. Closure status',
    '**AN7 is not automatically closed by report generation.** Closure requires manual review of SQL execution evidence, reconciliations, effect estimates, sensitivity results, leakage controls, limitations, and approved decisions.',
    ''
])
report_text='\n'.join(report_lines)
report_path=PROJECT_ROOT/'outputs'/'an7'/'AN7_final_report.md'
report_path.parent.mkdir(parents=True,exist_ok=True)
report_path.write_text(report_text,encoding='utf-8')
print('AN7 evidence report written to:',report_path)
display(Markdown(report_text))